# Notebook 02 — Preprocessing: Face Crops + Audio Extraction

**Goal:** For each clip in the manifest, extract two tensors:
- **Face crops:** `[16, 3, 112, 112]` uint8 — 16 uniformly-sampled frames, each cropped to a 112×112 face region using MTCNN
- **Audio:** `[160000]` float32 — exactly 10 seconds of mono audio at 16 kHz (pad with zeros if shorter, center-crop if longer)

Saved together as a single `.pt` dict per clip:
```python
{
    'faces': torch.uint8 [16, 3, 112, 112],
    'audio': torch.float32 [160000],
    'rel_path': str,
    'label': 'REAL' | 'FAKE',
    'split': 'train' | 'val' | 'test',
    'face_detection_info': dict (detected/fallback/no-face counts)
}
```

**Resumability:** clips with existing output files are skipped. Re-running the notebook continues from where it left off.

**Runtime estimate:** ~45-60 minutes for 6,529 clips on RTX A6000 (MTCNN on GPU is the bottleneck — about 0.4 sec per clip).

**Disk impact:** ~5-7 GB output (`data/preprocessed/*.pt`).

## 1. Imports and configuration

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import cv2
from PIL import Image
from tqdm.auto import tqdm

import av
import librosa
from facenet_pytorch import MTCNN

# Suppress noisy warnings from facenet-pytorch and pyav
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print(f"Python:    {sys.version.split()[0]}")
print(f"PyTorch:   {torch.__version__}")
print(f"CUDA OK:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:       {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path('..').resolve()

DATA_DIR          = PROJECT_ROOT / 'data'
CELEB_DIR         = DATA_DIR / 'celeb-df'
PREPROCESSED_DIR  = DATA_DIR / 'preprocessed'
MANIFEST_CSV      = DATA_DIR / 'manifest.csv'
FAILURE_LOG_CSV   = DATA_DIR / 'preprocessing_failures.csv'

# Video preprocessing
NUM_FRAMES        = 16     # frames per clip for ViViT-B/16
FACE_SIZE         = 112    # face crop resolution
FACE_MARGIN_PCT   = 0.20   # bbox expansion factor before cropping
MTCNN_MIN_CONF    = 0.90   # minimum detection probability to accept

# Audio preprocessing
AUDIO_SR          = 16000  # WavLM expects 16 kHz mono
AUDIO_DURATION_S  = 10     # 10 seconds
AUDIO_LENGTH      = AUDIO_SR * AUDIO_DURATION_S  # 160,000 samples

# Compute
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"PROJECT_ROOT:    {PROJECT_ROOT}")
print(f"Manifest:        {MANIFEST_CSV}")
print(f"Output dir:      {PREPROCESSED_DIR}")
print(f"Frames/clip:     {NUM_FRAMES}")
print(f"Face size:       {FACE_SIZE}x{FACE_SIZE}")
print(f"Audio length:    {AUDIO_LENGTH:,} samples ({AUDIO_DURATION_S}s @ {AUDIO_SR}Hz)")
print(f"Device:          {DEVICE}")

## 2. Folder setup and manifest load

In [ ]:
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for subfolder in ['Celeb-real', 'Celeb-synthesis', 'YouTube-real']:
    (PREPROCESSED_DIR / subfolder).mkdir(parents=True, exist_ok=True)

if not MANIFEST_CSV.exists():
    raise FileNotFoundError(f"Manifest not found at {MANIFEST_CSV}. Run notebook 01 first.")

manifest_df = pd.read_csv(MANIFEST_CSV)
print(f"Manifest loaded: {len(manifest_df):,} clips")
print(f"\nSplit counts:")
print(manifest_df['split'].value_counts())
print(f"\nLabel counts:")
print(manifest_df['label'].value_counts())

## 3. Initialize MTCNN

We use `facenet-pytorch`'s MTCNN (Zhang et al., 2016) — the standard face detector for deepfake detection papers, including AV-XAI's reference implementation.

We use the `.detect()` API (returns bounding boxes + confidences) rather than the high-level forward API, because we want explicit control over the fallback strategy when no face is detected in a frame.

In [ ]:
mtcnn = MTCNN(
    keep_all=True,         # return all detected faces; we pick the best
    min_face_size=40,      # ignore tiny detections (likely noise)
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=False,
    device=DEVICE,
)
print(f"MTCNN initialized on {DEVICE}")
print(f"  min_face_size: 40 pixels")
print(f"  min confidence to accept: {MTCNN_MIN_CONF}")

## 4. Frame sampling + face detection

For each clip:
1. Open the video with OpenCV, get total frame count
2. Compute 16 uniformly-spaced frame indices (if video has <16 frames, repeat last frame)
3. Read those frames, convert BGR → RGB
4. Run MTCNN on all 16 frames as a single batch (one GPU call)
5. For each frame, pick the highest-confidence face above threshold
6. Crop with margin, resize to 112×112, store as uint8 tensor

**Fallback strategy** when a frame has no detected face:
- Use the most recently-detected face's bbox from a nearby frame (maintains spatial consistency)
- If no face has been detected yet in the clip, fall back to a center crop

This keeps the face sequence spatially coherent even when MTCNN misses a frame.

In [ ]:
def read_uniform_frames(video_path: Path, num_frames: int = NUM_FRAMES) -> list[np.ndarray] | None:
    """Read `num_frames` uniformly-spaced frames from video, in RGB.
    Returns list of [H, W, 3] numpy arrays, or None if video unreadable.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 1:
        cap.release()
        return None
    
    if total >= num_frames:
        indices = np.linspace(0, total - 1, num_frames).astype(int)
    else:
        indices = list(range(total)) + [total - 1] * (num_frames - total)
    
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            if frames:
                frames.append(frames[-1])
            else:
                cap.release()
                return None
            continue
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def detect_and_crop(frames: list[np.ndarray]) -> tuple[torch.Tensor, dict]:
    """Run MTCNN on a batch of frames, return face crops with smart fallback.
    
    Returns:
        faces: torch.uint8 [N, 3, FACE_SIZE, FACE_SIZE]
        info:  dict with detected/fallback/center counts
    """
    pil_images = [Image.fromarray(f) for f in frames]
    
    # Batch detect: lists of bboxes [N_faces, 4] and probs [N_faces] per image
    boxes_list, probs_list = mtcnn.detect(pil_images)
    
    last_good_box = None
    crops = []
    detected = fallback = centered = 0
    
    for frame, boxes, probs in zip(frames, boxes_list, probs_list):
        h, w = frame.shape[:2]
        chosen_box = None
        
        if boxes is not None and len(boxes) > 0:
            # Filter by confidence threshold
            valid_mask = probs >= MTCNN_MIN_CONF
            if valid_mask.any():
                valid_probs = probs[valid_mask]
                valid_boxes = boxes[valid_mask]
                # Highest confidence face
                best = int(np.argmax(valid_probs))
                chosen_box = valid_boxes[best]
                last_good_box = chosen_box
                detected += 1
        
        if chosen_box is None:
            if last_good_box is not None:
                chosen_box = last_good_box
                fallback += 1
            else:
                # No face detected anywhere in clip yet — use center square crop
                size = min(h, w) * 0.7  # central 70%
                cx, cy = w / 2, h / 2
                chosen_box = np.array([cx - size/2, cy - size/2, cx + size/2, cy + size/2])
                centered += 1
        
        # Expand by margin
        x1, y1, x2, y2 = chosen_box
        bw, bh = x2 - x1, y2 - y1
        m = max(bw, bh) * FACE_MARGIN_PCT
        x1, y1, x2, y2 = x1 - m, y1 - m, x2 + m, y2 + m
        
        # Clamp to frame bounds
        x1 = max(0, int(x1)); y1 = max(0, int(y1))
        x2 = min(w, int(x2)); y2 = min(h, int(y2))
        
        # Crop and resize
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            crop = frame  # safety fallback
        crop = cv2.resize(crop, (FACE_SIZE, FACE_SIZE), interpolation=cv2.INTER_AREA)
        
        # To tensor [3, H, W] uint8
        crops.append(torch.from_numpy(crop).permute(2, 0, 1).contiguous())
    
    faces_tensor = torch.stack(crops)  # [N, 3, H, W] uint8
    return faces_tensor, {
        'detected': detected, 'fallback': fallback, 'centered': centered,
    }

## 5. Audio extraction (PyAV + librosa)

PyAV ships its own ffmpeg libraries — no need for a system ffmpeg install. Process:
1. Open the video container, locate the audio stream
2. Decode all audio frames into a numpy array
3. Mix down to mono if stereo
4. Resample to 16 kHz using librosa
5. Pad with zeros (if <10s) or center-crop (if >10s) to exactly 160,000 samples

In [ ]:
def extract_audio(video_path: Path) -> tuple[torch.Tensor, bool]:
    """Extract audio, resample to 16 kHz mono, pad/crop to 10 seconds.
    
    Returns:
        audio: torch.float32 [AUDIO_LENGTH]
        has_audio: bool (False means we returned silence)
    """
    try:
        container = av.open(str(video_path))
        audio_streams = [s for s in container.streams if s.type == 'audio']
        if not audio_streams:
            container.close()
            return torch.zeros(AUDIO_LENGTH, dtype=torch.float32), False
        
        audio_stream = audio_streams[0]
        orig_sr = audio_stream.rate
        
        # Decode all audio frames into one continuous array
        chunks = []
        for frame in container.decode(audio=0):
            arr = frame.to_ndarray()  # [channels, samples] or [samples] depending
            if arr.ndim == 1:
                arr = arr[np.newaxis, :]
            chunks.append(arr)
        container.close()
        
        if not chunks:
            return torch.zeros(AUDIO_LENGTH, dtype=torch.float32), False
        
        audio = np.concatenate(chunks, axis=1)  # [channels, total_samples]
        
        # Mono mixdown
        if audio.shape[0] > 1:
            audio = audio.mean(axis=0)
        else:
            audio = audio[0]
        
        audio = audio.astype(np.float32)
        
        # Normalize int → float if needed (PyAV may return int16/int32)
        if np.abs(audio).max() > 2.0:
            audio = audio / 32768.0  # int16 range
        
        # Resample to 16 kHz
        if orig_sr != AUDIO_SR:
            audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=AUDIO_SR)
        
        # Pad or center-crop to fixed length
        if len(audio) < AUDIO_LENGTH:
            audio = np.concatenate([audio, np.zeros(AUDIO_LENGTH - len(audio), dtype=np.float32)])
        elif len(audio) > AUDIO_LENGTH:
            start = (len(audio) - AUDIO_LENGTH) // 2
            audio = audio[start:start + AUDIO_LENGTH]
        
        return torch.from_numpy(audio).float(), True
    except Exception as e:
        return torch.zeros(AUDIO_LENGTH, dtype=torch.float32), False

## 6. Single-clip processing function

Combines video frame sampling + face detection + audio extraction. Returns a dict ready to be `torch.save`'d.

In [ ]:
def process_clip(row: pd.Series) -> tuple[dict | None, str | None]:
    """Process a single clip; return (output_dict, error_message).
    
    output_dict is None and error_message is a string if processing failed.
    """
    video_path = CELEB_DIR / row['rel_path']
    if not video_path.exists():
        return None, f"Video file missing: {video_path}"
    
    # ---- Video: sample 16 frames ----
    frames = read_uniform_frames(video_path)
    if frames is None:
        return None, "Could not read video"
    
    # ---- Video: detect faces ----
    try:
        faces, face_info = detect_and_crop(frames)
    except Exception as e:
        return None, f"MTCNN error: {type(e).__name__}: {e}"
    
    # ---- Audio ----
    audio, has_audio = extract_audio(video_path)
    
    return {
        'faces': faces,                   # uint8 [16, 3, 112, 112]
        'audio': audio,                   # float32 [160000]
        'rel_path': row['rel_path'],
        'label': row['label'],
        'split': row['split'],
        'source_id': row['source_id'],
        'target_id': row['target_id'] if pd.notna(row.get('target_id')) else None,
        'face_detection_info': face_info,
        'has_audio': has_audio,
    }, None


def output_path_for(rel_path: str) -> Path:
    """Compute the output .pt path for a given clip relative path."""
    return PREPROCESSED_DIR / (rel_path[:-4] + '.pt')  # strip .mp4, add .pt

## 7. Smoke test — process one clip manually

Before running the full loop, run on one clip to verify everything works end-to-end. This catches dependency/library issues fast (better to find them on clip 1 than clip 6000).

In [ ]:
# Pick the first train clip for smoke testing
smoke_row = manifest_df[manifest_df['split'] == 'train'].iloc[0]
print(f"Smoke testing on: {smoke_row['rel_path']}")

import time
t0 = time.time()
output, err = process_clip(smoke_row)
elapsed = time.time() - t0

if err:
    print(f"[FAIL] {err}")
else:
    print(f"[OK] Processed in {elapsed:.2f}s")
    print(f"  faces:   shape={tuple(output['faces'].shape)}, dtype={output['faces'].dtype}, range=[{output['faces'].min().item()}, {output['faces'].max().item()}]")
    print(f"  audio:   shape={tuple(output['audio'].shape)}, dtype={output['audio'].dtype}, range=[{output['audio'].min().item():.3f}, {output['audio'].max().item():.3f}]")
    print(f"  face detection: {output['face_detection_info']}")
    print(f"  has_audio: {output['has_audio']}")
    print(f"\nProjected total runtime: {elapsed * len(manifest_df) / 60:.0f} minutes for {len(manifest_df):,} clips")

## 8. Main loop — process all clips

Iterates through every clip in the manifest. Skips any clip whose output `.pt` already exists (idempotent: safe to interrupt and re-run).

Failed clips are logged to `data/preprocessing_failures.csv` rather than crashing the whole run.

In [ ]:
failures = []
skipped = 0
processed = 0

pbar = tqdm(manifest_df.itertuples(index=False), total=len(manifest_df), desc="Preprocessing")
for row in pbar:
    row = row._asdict()
    out_path = output_path_for(row['rel_path'])
    
    if out_path.exists():
        skipped += 1
        continue
    
    output, err = process_clip(pd.Series(row))
    
    if err:
        failures.append({'rel_path': row['rel_path'], 'error': err})
        continue
    
    out_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(output, out_path)
    processed += 1
    
    # Periodic progress refresh in postfix
    if processed % 50 == 0:
        pbar.set_postfix(processed=processed, skipped=skipped, failed=len(failures))

pbar.close()

# Save failure log
if failures:
    pd.DataFrame(failures).to_csv(FAILURE_LOG_CSV, index=False)
    print(f"\n[!] {len(failures)} clips failed. See {FAILURE_LOG_CSV}")
    print(f"Example failures:")
    for f in failures[:5]:
        print(f"  {f['rel_path']}: {f['error']}")

print(f"\n--- Preprocessing complete ---")
print(f"  Newly processed: {processed:,}")
print(f"  Previously done (skipped): {skipped:,}")
print(f"  Failed: {len(failures):,}")
print(f"  Total successful: {processed + skipped:,} / {len(manifest_df):,}")

## 9. Verification — sanity check the outputs

In [ ]:
# Count outputs per folder and split
all_outputs = list(PREPROCESSED_DIR.rglob('*.pt'))
print(f"Total .pt files on disk: {len(all_outputs):,}")

# Per-folder counts
from collections import Counter
folder_counts = Counter(p.parent.name for p in all_outputs)
print("\nBy folder:")
for folder, count in sorted(folder_counts.items()):
    expected = (manifest_df['folder'] == folder).sum()
    print(f"  {folder}: {count:,} / {expected:,}")

# Total disk usage
total_size_gb = sum(p.stat().st_size for p in all_outputs) / (1024**3)
avg_size_kb = total_size_gb * 1024**2 / max(1, len(all_outputs))
print(f"\nDisk usage:")
print(f"  Total:   {total_size_gb:.2f} GB")
print(f"  Avg/clip: {avg_size_kb:.0f} KB")

In [ ]:
# Load a few random samples and check their structure
import random
random.seed(0)
samples = random.sample(all_outputs, min(5, len(all_outputs)))

print("--- Sample inspection ---")
for path in samples:
    data = torch.load(path, weights_only=False)
    print(f"\n{path.relative_to(PREPROCESSED_DIR)}:")
    print(f"  faces:    {tuple(data['faces'].shape)} {data['faces'].dtype}")
    print(f"  audio:    {tuple(data['audio'].shape)} {data['audio'].dtype}")
    print(f"  label:    {data['label']}")
    print(f"  split:    {data['split']}")
    print(f"  face info: {data['face_detection_info']}")
    print(f"  has_audio: {data['has_audio']}")

In [ ]:
# Aggregate detection statistics across all processed clips
# This tells us how often MTCNN failed and we used fallbacks
print("Aggregating face detection stats across all outputs (may take 30s)...")
detected_total = fallback_total = centered_total = 0
no_audio_count = 0

for path in tqdm(all_outputs, desc="Reading metadata"):
    data = torch.load(path, weights_only=False)
    info = data['face_detection_info']
    detected_total += info['detected']
    fallback_total += info['fallback']
    centered_total += info['centered']
    if not data.get('has_audio', True):
        no_audio_count += 1

total_frames = detected_total + fallback_total + centered_total
if total_frames > 0:
    print(f"\nFace detection breakdown across {total_frames:,} frames:")
    print(f"  Detected:        {detected_total:,} ({detected_total/total_frames*100:.1f}%)")
    print(f"  Fallback (prev bbox): {fallback_total:,} ({fallback_total/total_frames*100:.1f}%)")
    print(f"  Center crop:     {centered_total:,} ({centered_total/total_frames*100:.1f}%)")

print(f"\nClips without audio track: {no_audio_count:,} / {len(all_outputs):,}")

## 10. Visualize sample face crops

Look at a few preprocessed clips to make sure the face crops actually contain faces (not backgrounds or skewed crops). Pick one REAL and one FAKE.

In [ ]:
import matplotlib.pyplot as plt

def show_clip_frames(pt_path: Path, n_show: int = 8):
    data = torch.load(pt_path, weights_only=False)
    faces = data['faces'].numpy()  # [16, 3, 112, 112] uint8
    # Take n_show evenly-spaced frames from the 16
    indices = np.linspace(0, faces.shape[0] - 1, n_show).astype(int)
    
    fig, axes = plt.subplots(1, n_show, figsize=(n_show * 1.5, 1.8))
    for i, idx in enumerate(indices):
        img = faces[idx].transpose(1, 2, 0)  # [H, W, 3]
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(f'f{idx}', fontsize=8)
    fig.suptitle(f"{data['rel_path']}  |  {data['label']}  |  {data['split']}", fontsize=10)
    plt.tight_layout()
    plt.show()


# One REAL clip from Celeb-real
real_outputs = [p for p in all_outputs if p.parent.name == 'Celeb-real']
if real_outputs:
    show_clip_frames(real_outputs[0])

# One FAKE clip from Celeb-synthesis
fake_outputs = [p for p in all_outputs if p.parent.name == 'Celeb-synthesis']
if fake_outputs:
    show_clip_frames(fake_outputs[0])

# One from YouTube-real
yt_outputs = [p for p in all_outputs if p.parent.name == 'YouTube-real']
if yt_outputs:
    show_clip_frames(yt_outputs[0])

## 11. Visualize sample audio waveform

Quick check that audio actually got extracted properly. If everything is silence, something is wrong.

In [ ]:
sample_path = real_outputs[0] if real_outputs else all_outputs[0]
data = torch.load(sample_path, weights_only=False)
audio = data['audio'].numpy()

fig, ax = plt.subplots(1, 1, figsize=(12, 2.5))
t = np.arange(len(audio)) / AUDIO_SR
ax.plot(t, audio, linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f"{data['rel_path']}  |  has_audio={data['has_audio']}  |  RMS={np.sqrt((audio**2).mean()):.4f}")
ax.set_xlim(0, AUDIO_DURATION_S)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# RMS check: if it's near zero, audio extraction probably failed
rms = np.sqrt((audio**2).mean())
print(f"Audio RMS: {rms:.4f}")
if rms < 1e-6:
    print("  [WARN] Audio appears to be silence — check audio extraction")
else:
    print("  [OK] Audio signal is present")

## 12. Next steps

If verification looks good (face crops look face-like, audio waveform has signal, failure count is low):

1. **Update `project_log.md`** under Section 2 (Preprocessing log):
   - Total clips successfully preprocessed: N
   - Failed clips: N (with reasons if known)
   - Face detection success rate (% of frames with direct MTCNN detection)
   - Disk size of preprocessed data

2. **Commit + push** the notebook and project log update.

3. **Move on to notebook 03** (`03_model_definition.ipynb`) — define the AV-XAI architecture: ViViT video encoder + WavLM audio encoder + cross-modal consistency module + hierarchical fusion + classifier heads.

**Optional disk cleanup** if space is tight: once preprocessing is complete and verified, the raw `data/celeb-df/*.mp4` files are no longer needed for training (~10 GB recovery). Keep them for now in case we need to re-preprocess with different parameters.